In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
# Averaged spectrum from capture_spectrum() (results/vco/vco_capture_*.csv); runs on its own, without the cells below.
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

CAPTURE_CSV = None  # e.g. Path.cwd().resolve().parents[2] / "results" / "vco" / "vco_capture_S5_20261005_120000.csv"; None = newest

if CAPTURE_CSV is None:
    CAPTURE_CSV = sorted((Path.cwd().resolve().parents[2] / "results" / "vco").glob("vco_capture_*.csv"),
                         key=lambda p: p.stat().st_mtime)[-1]
CAPTURE_CSV = Path(CAPTURE_CSV)
cap = pd.read_csv(CAPTURE_CSV)
meta = cap.iloc[0]
peak = cap.loc[cap["level_dbm"].idxmax()]
setting = (f"{meta['f_target'] / 1e6:g} MHz target: c{meta['vco_tune_coarse']:.0f} min {meta['vco_current_min']:.0f} "
           f"max {meta['vco_current_max']:.0f}, Vctrl {meta['vctrl_meas']:.3f} V, /{meta['total_div']:.0f}")
print(f"{CAPTURE_CSV.name}: {setting}")
print(f"{len(cap)} points, {meta['averages']:.0f} sweeps averaged, RBW {meta['rbw']:g} Hz; peak {peak['level_dbm']:.1f} dBm "
      f"at {peak['freq'] / 1e6:.4f} MHz on the pad -> f_vco {peak['freq'] * meta['total_div'] / 1e6:.2f} MHz")

fig = go.Figure()
fig.add_scatter(x=cap["freq"] / 1e6, y=cap["level_dbm"], mode="lines", line=dict(color="#2a78d6", width=1.5),
                hovertemplate="%{x:.4f} MHz: %{y:.1f} dBm<extra></extra>")
fig.update_layout(title=f"Averaged spectrum on the pad (= PLL output) ({setting})<br><sup>center {meta['center'] / 1e6:g} MHz, "
                        f"span {meta['span'] / 1e6:g} MHz, RBW {meta['rbw']:g} Hz, {meta['averages']:.0f} averages</sup>",
                  template="plotly_white", height=500)
fig.update_xaxes(title="frequency [MHz]", gridcolor="#e4e3df")
fig.update_yaxes(title="level [dBm]", gridcolor="#e4e3df")
base = CAPTURE_CSV.with_suffix("")
fig.write_html(f"{base}.html")
try:
    fig.write_image(f"{base}.png", width=1100, height=500, scale=2)
    print(f"saved {base}.html and .png")
except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
    print(f"saved {base}.html; PNG export failed: {e}")
fig.show()

vco_capture_S5_20261005_110703.csv: 2400 MHz target: c2 min 9 max 12, Vctrl 0.499 V, /64
691 points, 20 sweeps averaged, RBW 200000 Hz; peak -3.0 dBm at 36.5870 MHz on the pad -> f_vco 2341.57 MHz
saved /users/micas/wvandest/projects/lagd-meas/results/vco/vco_capture_S5_20261005_110703.html and .png


In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

VCO_DIR = Path.cwd().resolve().parents[2] / "results" / "vco"  # notebook lives in sw/tools/notebooks/
SPECTRUM_CSV = None  # e.g. VCO_DIR / "vco_spectrum_S5_20261002_150000.csv"; None = newest run

if SPECTRUM_CSV is None:
    SPECTRUM_CSV = sorted((p for p in VCO_DIR.glob("vco_spectrum_*.csv") if not p.stem.endswith("_traces")),
                          key=lambda p: p.stat().st_mtime)[-1]
SPECTRUM_CSV = Path(SPECTRUM_CSV)
TRACE_CSV = SPECTRUM_CSV.with_name(SPECTRUM_CSV.stem + "_traces.csv")
runs = pd.read_csv(SPECTRUM_CSV).sort_values("f_target").reset_index(drop=True)
traces = pd.read_csv(TRACE_CSV) if TRACE_CSV.exists() else pd.DataFrame(columns=["f_target"])
ok = runs[runs["status"] == "ok"]
offsets = [c[len("L_vco_"):] for c in runs.columns if c.startswith("L_vco_")]
print(f"{SPECTRUM_CSV.name}: {len(runs)} settings, {len(ok)} ok; traces: {TRACE_CSV.name if len(traces) else 'none'}")

pd.DataFrame({
    "f_target [MHz]": runs["f_target"] / 1e6,
    "setting": [f"c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}"
                if not np.isnan(r.vco_tune_coarse) else "-" for r in runs.itertuples()],
    "Vctrl [V]": runs["vctrl_meas"],
    "|Kvco| pred [MHz/V]": runs["kvco_pred"] / 1e6,
    "N": runs["total_div"],
    "f_vco meas [MHz]": runs["f_vco_meas"] / 1e6,
    "f dev [%]": runs["f_dev_pct"],
    **{f"L_vco @ {float(o):g} Hz [dBc/Hz]": runs[f"L_vco_{o}"] for o in offsets},
    "jitter [ps]": runs["time_jitter_ps"],
    "phase jitter VCO [mrad]": runs["phase_jitter_vco_mrad"],
    "status": runs["status"],
}).round(3)

vco_spectrum_S5_20261002_172235.csv: 1 settings, 1 ok; traces: vco_spectrum_S5_20261002_172235_traces.csv


,f_target [MHz],setting,Vctrl [V],|Kvco| pred [MHz/V],N,f_vco meas [MHz],f dev [%],L_vco @ 10000 Hz [dBc/Hz],L_vco @ 100000 Hz [dBc/Hz],L_vco @ 500000 Hz [dBc/Hz],L_vco @ 1e+06 Hz [dBc/Hz],jitter [ps],phase jitter VCO [mrad],status
0,1000.0,c7 min 11 max 14,0.485,1030.981,32,879.215,-12.078,16.422,16.033,3.898,1.703,743536.98,4107501.031,ok


In [10]:
INK, GRID = "#52514e", "#e4e3df"
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
# Frequency axes (values in MHz, log): labelled decades with the unit in the tick, minor grid in between.
FREQ_AXIS = dict(type="log", tickvals=[1, 10, 100, 1e3, 1e4, 1e5],
                 ticktext=["1 MHz", "10 MHz", "100 MHz", "1 GHz", "10 GHz", "100 GHz"],
                 minor=dict(showgrid=True, gridcolor="#f1f0ed"))
OFFSET_AXIS = dict(type="log", tickvals=[1e3, 1e4, 1e5, 1e6, 1e7], ticktext=["1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))


def freq_label(f_hz):
    return f"{f_hz / 1e9:g} GHz" if f_hz >= 1e9 else f"{f_hz / 1e6:g} MHz"


# Colour per setting, fixed by its position in the run (not cycled: more than 8 settings -> split the run).
COLORS = {f: PALETTE[i] for i, f in enumerate(ok["f_target"])}
NAMES = {r.f_target: f"{freq_label(r.f_target)}: c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} "
                     f"max {r.vco_current_max:.0f}" for r in ok.itertuples()}


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = SPECTRUM_CSV.with_name(f"{SPECTRUM_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [11]:
# Averaged analyzer traces per span, relative to the carrier (trace maximum), in the RBW of that span.
spans = sorted(traces["span"].unique()) if len(traces) else []
fig = make_subplots(rows=1, cols=max(len(spans), 1), horizontal_spacing=0.05,
                    subplot_titles=[f"span {s / 1e3:g} kHz (RBW {traces.loc[traces['span'] == s, 'rbw'].iloc[0]:g} Hz)"
                                    for s in spans])
for col, span in enumerate(spans, 1):
    for f_t, d in traces[traces["span"] == span].groupby("f_target"):
        if f_t not in COLORS:
            continue
        fig.add_scatter(x=d["offset"] / 1e3, y=d["power_dbm"] - d["power_dbm"].max(), mode="lines",
                        line=dict(color=COLORS[f_t], width=1.5), name=NAMES[f_t], legendgroup=NAMES[f_t],
                        showlegend=col == 1, row=1, col=col,
                        hovertemplate=f"{NAMES[f_t]}<br>%{{x:.2f}} kHz: %{{y:.1f}} dBc<extra></extra>")
fig.update_layout(title="Spectrum on the pad (= PLL output) around the carrier", template="plotly_white", height=500,
                  legend=dict(orientation="h", y=-0.2), hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="offset from the carrier [kHz]", gridcolor=GRID)
fig.update_yaxes(gridcolor=GRID)
fig.update_yaxes(title="power relative to the carrier [dBc]", col=1)
save(fig, "traces")

saved /users/micas/wvandest/projects/lagd-meas/results/vco/vco_spectrum_S5_20261002_172235_traces.html and .png


In [12]:
# VCO-referred phase noise: the phase-noise marker (circles) and an estimate from the traces (lines):
# P(offset) - P(carrier) - 10 log10(RBW) + 20 log10(N), upper sideband, offsets > 20 RBW (no ENBW correction).
fig = go.Figure()
for r in ok.itertuples():
    f_t, n = r.f_target, r.total_div
    d = traces[(traces["f_target"] == f_t)]
    for span, t in d.groupby("span"):
        t = t[t["offset"] > 20 * t["rbw"]]
        l_est = t["power_dbm"] - d.loc[d["span"] == span, "power_dbm"].max() - 10 * np.log10(t["rbw"]) + 20 * np.log10(n)
        fig.add_scatter(x=t["offset"], y=l_est, mode="lines", line=dict(color=COLORS[f_t], width=1), opacity=0.6,
                        legendgroup=NAMES[f_t], showlegend=False,
                        hovertemplate=f"{NAMES[f_t]} (trace)<br>%{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz<extra></extra>")
    fig.add_scatter(x=[float(o) for o in offsets], y=[getattr(r, f"L_vco_{o}", np.nan) for o in offsets],
                    mode="markers", marker=dict(color=COLORS[f_t], size=10, line=dict(color="white", width=1)),
                    name=NAMES[f_t], legendgroup=NAMES[f_t],
                    hovertemplate=f"{NAMES[f_t]} (marker)<br>%{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz<extra></extra>")
fig.update_layout(title="Phase noise referred to the VCO (pad + 20 log10 N)", template="plotly_white", height=550,
                  legend=dict(orientation="h", y=-0.18), hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **OFFSET_AXIS)
fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "phase_noise")

saved /users/micas/wvandest/projects/lagd-meas/results/vco/vco_spectrum_S5_20261002_172235_phase_noise.html and .png


In [13]:
# RMS time jitter over the marker offsets, per measured VCO frequency.
fig = go.Figure()
fig.add_scatter(x=ok["f_vco_meas"] / 1e6, y=ok["time_jitter_ps"], mode="markers+text",
                marker=dict(color=[COLORS[f] for f in ok["f_target"]], size=11, line=dict(color="white", width=1)),
                text=[f"c{c:.0f}" for c in ok["vco_tune_coarse"]], textposition="top center",
                textfont=dict(color=INK, size=10), showlegend=False,
                customdata=np.c_[ok["phase_jitter_vco_mrad"], ok["kvco_pred"] / 1e6, ok["vctrl_meas"]],
                hovertemplate=("f_vco %{x:.1f} MHz: %{y:.3f} ps<br>%{customdata[0]:.1f} mrad at the VCO<br>"
                               "|Kvco| %{customdata[1]:.0f} MHz/V at Vctrl %{customdata[2]:.3f} V<extra></extra>"))
lo, hi = ok["offset_min"].min(), ok["offset_max"].max()
fig.update_layout(title=f"RMS time jitter (same on the pad and at the VCO), integrated {lo / 1e3:g} kHz - {hi / 1e6:g} MHz",
                  template="plotly_white", height=500)
fig.update_xaxes(title="f_vco", gridcolor=GRID, **FREQ_AXIS)
fig.update_yaxes(title="RMS time jitter [ps]", gridcolor=GRID, rangemode="tozero")
save(fig, "jitter")

saved /users/micas/wvandest/projects/lagd-meas/results/vco/vco_spectrum_S5_20261002_172235_jitter.html and .png
